In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
from torchvision.transforms.v2.functional import to_pil_image

%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import torch.nn.functional as F
from common import *

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
import os

def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
ratios = [0.05]
lambdas = 0.05
loss = []
acc_object = []
acc_background = []
steps = []
eig_list = []
method = "BarlowTwins_single_background"
directory_models = f"/models/jhkim/waterbirds/{method}_{lambdas}"
directory = f"datas/{method}_{lambdas}"
for r in ratios:
    loss_array = load_from_csv(f"{directory}/{r}/loss.csv")
    [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{r}/accuracy.csv")
    eigvals_lists_array = load_from_csv_split(f"{directory}/{r}/eig.csv")
    loss.append(loss_array)
    acc_object.append(acc_object_array)
    acc_background.append(acc_background_array)
    steps.append(step_array)
    eig_list.append(eigvals_lists_array)

In [ ]:
background_predictability = 70

fig, axs = plt.subplots(1, 2, figsize=(28, 10))

# plot_range = 300

positions = []
plateau_range = 2.5
for r in range(len(ratios)):
    pos_low = np.where(acc_background[r] > 99)
    pos = np.where(acc_object[r] > background_predictability + plateau_range)
    if len(pos[0]) > 0:
        positions.append(pos[0][0])
        axs[0].axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
        axs[1].axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
    axs[0].plot(steps[r], acc_object[r], alpha=0.7, linewidth=5, color=colors[r],
                label=f"Object ratio : {ratios[r]:.3f}")
    axs[0].plot(steps[r], acc_background[r], linestyle="--", alpha=0.75, linewidth=4, color=colors[r*2], label=f"Object ratio : {ratios[r]:.3f}")
    print(steps[r][positions[-1]])
    print(steps[r][pos_low[0][0]])

upper_y = np.zeros_like(steps[0]) + background_predictability + plateau_range
lower_y = np.zeros_like(steps[0]) + background_predictability - plateau_range
axs[0].fill_between(steps[0], upper_y, lower_y, color='skyblue', alpha=0.4)
# axs[0].text(500, 67, "bg correlation")

# axs.set_xlim(0-1, epochs)
# axs.set_ylim(10, 100)
axs[0].axhline(background_predictability, linestyle="--", color="black", linewidth=2, alpha=0.4)
# class AnyObjectHandler(HandlerBase):
#     def create_artists(self, legend, orig_handle,
#                        x0, y0, width, height, fontsize, trans):
#         l1 = plt.Line2D([x0,y0+width], [0.7*height,0.7*height],
#                                                 linestyle='--', color=colors[r*2])
#         l2 = plt.Line2D([x0,y0+width], [0.3*height,0.3*height], color=colors[r*2+1])
#         return [l1, l2]
# plt.legend([object], [f'Object ratio : {ratios[0]:.2f}'],  handler_map={object: AnyObjectHandler()})
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Test Accuracy")
# axs[0].set_ylim(60, 80)
# plt.tight_layout()
# plt.savefig("figure/loss_acc.pdf")

for r in range(len(ratios)):
    axs[1].plot(loss[r], label=f"ratio: {ratios[r]}", color=colors[r], alpha=0.5, linewidth=4)
    axs[1].plot(smooth_data(loss[r], window_size=10), label=f"ratio: {ratios[r]}", color=colors[r], alpha=1.0, linewidth=6)
    axs[1].set_xlabel("Step")
    axs[1].set_ylabel("Loss")
# plt.ylim(100, 150)
axs[1].set_yscale('log')
axs[0].legend(ncols=2, loc='lower right')

plt.tight_layout()
plt.savefig('figure/waterbirds_single_background.pdf', format='pdf')
plt.show()

In [ ]:
for r in range(len(ratios)):
    plt.plot(loss[r], label=f"ratio: {ratios[r]}", color=colors[r], alpha=0.6, linewidth=3)
    plt.xlabel("Step")
    plt.ylabel("Loss")
# plt.ylim(100, 300)
# plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
import torchvision.transforms.v2 as transforms
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from PIL.Image import Image
from PIL import Image, ImageOps, ImageFilter
import pandas as pd
import torchvision
import cv2

device = torch.device("cuda")
size = 96


class WaterbirdDataset(Dataset):
    def __init__(self, root_dir, metadata_path, split='train', transform=None, balance_classes=True, no_aug=False):
        self.root_dir = root_dir
        self.transform = transform
        self.split = split
        self.no_aug = no_aug
        self.metadata = pd.read_csv(metadata_path)

        split_dict = {'train': 0, 'val': 1, 'test': 2}

        self.metadata = self.metadata[self.metadata['split'] == split_dict[split]]
        if balance_classes:
            landbird_data = self.metadata[self.metadata['y'] == 0]
            waterbird_data = self.metadata[self.metadata['y'] == 1]

            min_class_size = min(len(landbird_data), len(waterbird_data))
            # min_class_size = 256

            landbird_balanced = landbird_data.sample(min_class_size, random_state=1)
            waterbird_balanced = waterbird_data.sample(min_class_size, random_state=1)

            self.metadata = pd.concat([landbird_balanced, waterbird_balanced])

        self.metadata = self.metadata.reset_index(drop=True)

        self.size_transform = transforms.Resize(size)

        self.labels = []
        self.places = []
        self.img_ids = []
        self.img = []
        mean = 0.0
        var = 0.0

        for idx in range(len(self.metadata)):
            img_filename = self.metadata.loc[idx, 'img_filename']
            self.labels.append(int(self.metadata.loc[idx, 'y']))
            self.places.append(int(self.metadata.loc[idx, 'place']))
            self.img_ids.append(self.metadata.loc[idx, 'img_id'] if 'img_id' in self.metadata.columns else idx)
            image = Image.open(os.path.join(self.root_dir, img_filename)).convert('RGB')
            image_resize = self.size_transform(image)
            self.img.append(image_resize)
            image_preprocess = transforms.ToTensor()(image_resize)
            mean += image_preprocess.mean(dim=[1, 2])
            var += image_preprocess.var(dim=[1, 2])

        mean /= len(self.img)
        var /= len(self.img)
        std = torch.sqrt(var)
        print(mean, std)

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        self.transform_image = transforms.ToTensor()

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        image_original = self.img[idx]
        label = self.labels[idx]
        places = self.places[idx]
        if self.split == 'test':
            return self.transform(image_original), self.transform_image(image_original), label, places
        return self.transform(image_original), label, places

In [ ]:
class BarlowTwins(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torchvision.models.resnet34(zero_init_residual=True, weights=None)
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.size_in = self.backbone.fc.in_features
        self.size_out = 256
        self.backbone.fc = nn.Identity()

        sizes = [self.size_in, self.size_out, self.size_out]
        layers = []
        for i in range(len(sizes) - 2):
            layers.append(nn.Linear(sizes[i], sizes[i + 1], bias=False))
            layers.append(nn.BatchNorm1d(sizes[i + 1]))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Linear(sizes[-2], sizes[-1], bias=False))
        self.projector = nn.Sequential(*layers)

    def forward(self, x, use_projector=True):
        if not use_projector:
            return self.backbone(x)
        z = self.projector(self.backbone(x))
        return z

In [ ]:
def generate_resnet_from_dict(path):
    model = BarlowTwins()
    model.load_state_dict(torch.load(path))
    model.eval()
    return model, model.size_in

In [ ]:
ratio = 0.05

In [ ]:
root_dir = f"/home/dataset/waterbird_single_background/{ratio}"
metadata_path = f"/home/dataset/waterbird_single_background/{ratio}/metadata.csv"
set_seed(0)
train_noaug_dataset = WaterbirdDataset(
    root_dir=root_dir,
    metadata_path=metadata_path,
    split='train',
    balance_classes=True,
)
train_noaug_loader = DataLoader(
    train_noaug_dataset,
    batch_size=128,
    shuffle=True,
)

test_dataset = WaterbirdDataset(
    root_dir=root_dir,
    metadata_path=metadata_path,
    split='test',
    balance_classes=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
)

test_loader_batched = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
)

In [ ]:
# def cut_topk(x):
#     top_k = 1
#     u, s, vh = torch.linalg.svd(x, full_matrices=False)
#     s[top_k:] = 0
#     return u @ torch.diag(s) @ vh

def remove_background_subspace(out_early, out_late):
    # k = 10
    # u, s, vh = torch.linalg.svd(out_early, full_matrices=False)
    #
    # background_directions = vh[:k, :]  # [k, feature_dim]
    #
    # # proj = out_late @ V_k @ V_k^T
    # proj_to_background = (out_late @ background_directions.T) @ background_directions

    # cleaned = out_late - proj_to_background

    return out_late - out_early

In [ ]:
model_early, size_out = generate_resnet_from_dict(f"{directory_models}/{ratio}/80.pt")
model_early = model_early.to(device)
model_early.eval()
# model_late, _ = generate_resnet_from_dict(f"{directory_models}/{ratio}/520.pt")
model_late, _ = generate_resnet_from_dict(f"{directory_models}/{ratio}/1300.pt")
model_late = model_late.to(device)
model_late.eval()
# linear = nn.Sequential(nn.BatchNorm1d(size_out), nn.Linear(size_out, 2)).to(device)
# linear_proj = nn.Sequential(nn.BatchNorm1d(size_out), nn.Linear(size_out, 2)).to(device)
linear = nn.Linear(size_out, 2).to(device)
linear_proj = nn.Linear(size_out, 2).to(device)

linear_proj.load_state_dict(linear.state_dict())

optimizer_linear = torch.optim.Adam(linear.parameters())
optimizer_linear_proj = torch.optim.Adam(linear_proj.parameters())

linear.train()
linear_proj.train()
bar = tqdm(range(40))
for epoch in bar:
    loss_total = 0.0
    loss_total_proj = 0.0
    for image, label, _ in train_noaug_loader:
        with torch.no_grad():
            image = image.to(device)
            label = label.to(device)
            out_early = model_early(image, use_projector=False)
            out_late = model_late(image, use_projector=False)
            proj = remove_background_subspace(out_early, out_late)

        out_orig = linear(out_late)
        optimizer_linear.zero_grad()
        loss_linear = F.cross_entropy(out_orig, label)
        loss_linear.backward()
        loss_total += loss_linear.item()
        optimizer_linear.step()

        out_proj = linear_proj(proj)
        optimizer_linear_proj.zero_grad()
        loss_linear_proj = F.cross_entropy(out_proj, label)
        loss_linear_proj.backward()
        loss_total_proj += loss_linear_proj.item()
        optimizer_linear_proj.step()

    bar.set_description(f"loss : {loss_total / len(train_noaug_loader)}, loss_proj : {loss_total_proj / len(train_noaug_loader)}")

In [ ]:
linear.eval()
linear_proj.eval()
correct = 0.0
correct_proj = 0.0
total = 0.0

stats = {
    'landbird_land': {'orig': 0, 'clean': 0, 'total': 0},
    'landbird_water': {'orig': 0, 'clean': 0, 'total': 0},
    'waterbird_land': {'orig': 0, 'clean': 0, 'total': 0},
    'waterbird_water': {'orig': 0, 'clean': 0, 'total': 0},
}

with torch.no_grad():
    for image, _, label, background in test_loader_batched:
        image = image.to(device)
        label = label.to(device)
        out_early = model_early(image, use_projector=False)
        out_late = model_late(image, use_projector=False)
        proj = remove_background_subspace(out_early, out_late)
        out_def = linear(out_late)
        out_proj = linear_proj(proj)

        # correct += (torch.argmax(out_def, dim=1) == label).sum().item()
        # correct_proj += (torch.argmax(out_proj, dim=1) == label).sum().item()
        # total += label.size(0)
        pred_orig = torch.argmax(out_def, dim=1)
        pred_clean = torch.argmax(out_proj, dim=1)
        for i in range(len(label)):
            bird = 'landbird' if label[i] == 0 else 'waterbird'
            bg = 'land' if background[i] == 0 else 'water'
            key = f"{bird}_{bg}"

            stats[key]['total'] += 1
            stats[key]['orig'] += (pred_orig[i] == label[i]).item()
            stats[key]['clean'] += (pred_clean[i] == label[i]).item()

print(stats)
total = sum(stats[key]['total'] for key in stats)
print(f"acc : {(stats['landbird_land']['orig'] + stats['waterbird_water']['orig']) / total}, acc_proj : {(stats['landbird_land']['clean'] + stats['waterbird_water']['clean']) / total}")
total_correct_orig = sum(stats[key]['orig'] for key in stats)
total_correct_clean = sum(stats[key]['clean'] for key in stats)
print(f"total acc : {total_correct_orig / total * 100:.2f}%, "
      f"total acc_proj : {total_correct_clean / total * 100:.2f}%")

In [ ]:
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# target_layers = [model.layer1[-1], model.layer2[-1], model.layer3[-1], model.layer4[-1]]
target_layers = [model.layer4[-1]]
model_grad = nn.Sequential(model, linear).to(device)

count_ocean = 5
count_forest = 5
i_ocean = 0
i_forest = 0
images_list = [[], []]
for index, (image_tensor, image, label, background) in enumerate(test_loader):
    if label == background:
        continue
    if i_ocean >= count_ocean and i_forest >= count_forest:
        break
    if label == 0 and i_ocean >= count_ocean:
        continue
    if label == 1 and i_forest >= count_forest:
        continue
    # targets = [ClassifierOutputTarget(1)]
    targets = None
    current_image = image[0, :, :, :]
    current_image = current_image.permute(1, 2, 0)
    current_image = current_image.cpu().numpy()
    with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=image_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        visualization = show_cam_on_image(current_image, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs
    # plt.imshow(to_pil_image(current_image))
    # plt.show()
    # plt.imshow(to_pil_image(visualization))
    # plt.show()
    images_list[0].append(to_pil_image(current_image))
    images_list[1].append(to_pil_image(visualization))
    print(f"correct answer:{label.item()}, spurious answer: {background.item()}")
    print(f"model: {model_outputs.detach().cpu()}")
    if label == 0:
        i_ocean += 1
    else:
        i_forest += 1


In [ ]:
n_images = len(images_list[0])

fig, axes = plt.subplots(2, n_images, figsize=(3*n_images, 6))

for i in range(n_images):
    axes[0, i].imshow(images_list[0][i])
    axes[0, i].axis('off')

for i in range(n_images):
    axes[1, i].imshow(images_list[1][i])
    axes[1, i].axis('off')

plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.tight_layout(pad=0.1)
plt.show()